### Setup

In [1]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from common.utils import DataPreprocessor, FeatureEngineer, set_seed
from common.exp_data_utils import ExperimentDataPreprocessor
from common.eval import Evaluator

MOVIELENS_DATA_DIR = "../datasets/hetrec2011-movielens-2k-v2/user_ratedmovies.dat"
RANDOM_SEED = 42

# Initialize data processors
set_seed(RANDOM_SEED)
data_preprocessor = DataPreprocessor()
feature_engineer = FeatureEngineer()
experiment_data_preprocessor = ExperimentDataPreprocessor()
evaluator = Evaluator()


/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
Seed set to 42
Seed set to 42


random seed set to 42
numpy seed set to 42
torch seed set to 42
lightning seed set to 42
torch set to use deterministic algorithms


### Load and Process DataFrame

In [2]:
interaction_df = data_preprocessor.load_and_process_df(
    file_dir=MOVIELENS_DATA_DIR,
    year_range=(2006, 2008),
)
interaction_df.head()

Data count: 855598
Data count after filtering by year (2006, 2008): 480608
Num of distinct users: 2103
Num of distinct items: 9519
done!
------------------------------
Filtering by min user/item interactions (10/0):
Data count before: 480608
Data count after: 480448
done!
------------------------------
==== Final Data Info: ====
Data Year Range: (2006, 2008)
Rating Threshold: 4.0
Num of interactions: 480448
Num of distinct users: 2064
Num of distinct items: 9519


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1


### Join Side Information

In [3]:
interaction_info_df = data_preprocessor.join_item_features(
    df=interaction_df, actor_k=5, threshold=5,
)
interaction_info_df.head()

extracting item features...
merging features...
interaction data count before merging: 480448
interaction data count after merging: 478404
done!


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label,actorID,country,directorID,directorName,genre
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0,"[jack_lemmon, walter_matthau, annmargret, burg...",USA,donald_petrie,Donald Petrie,"[Comedy, Romance, [PAD], [PAD], [PAD], [PAD], ..."
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1,"[[RARE], [RARE], [RARE], [RARE], [RARE]]",USA,[RARE],Siddharth Randeria,"[Sci-Fi, Thriller, [PAD], [PAD], [PAD], [PAD],..."
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1,"[mel_gibson, sophie_marceau, patrick_mcgoohan,...",USA,[RARE],Mel Gibson,"[Action, Drama, War, [PAD], [PAD], [PAD], [PAD..."
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0,"[[RARE], laura_linney, ernie_hudson_jr, tim_cu...",USA,frank_marshall,Frank Marshall,"[Action, Adventure, Mystery, Sci-Fi, [PAD], [P..."
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1,"[antonio_banderas, salma_hayek, 1142520-joaqui...",USA,robert_rodriguez,Robert Rodriguez,"[Action, Romance, Thriller, [PAD], [PAD], [PAD..."


### Prepare Train/Valid/Test Set

In [4]:
# TODO: determine which method to use for splitting
# 1. Split by year
# 2. Stratified split by user, timestamp

train_df, valid_df, test_df = experiment_data_preprocessor.stratified_time_split(
    interaction_info_df,
    time_col="timestamp",
    train_ratio=0.75,
    val_ratio=0.1,
    test_ratio=0.15,
)

TRAIN_NUM_USERS = len(train_df["userID"].unique())
TRAIN_NUM_ITEMS = len(train_df["movieID"].unique())


Splitting data into train/valid/test by time period with ratio=(0.75 : 0.1 : 0.15):
train: 358027 (74.84%)
valid: 46916 (9.81%)
test: 73461 (15.36%)
------------------------------ 

Check target label distribution after splitting (%):
train label
0    0.555944
1    0.444056
Name: proportion, dtype: float64
valid label
0    0.610772
1    0.389228
Name: proportion, dtype: float64
test label
0    0.585277
1    0.414723
Name: proportion, dtype: float64


### Re-index User/Item ID & Encode Categorical Features

In [5]:
print("Train: fit_transform")
encoded_train_df = feature_engineer.fit_transform(train_df)
print("---"*10)
print("Valid: transform")
encoded_valid_df = feature_engineer.transform(valid_df)
print("---"*10)
print("Test: transform")
encoded_test_df = feature_engineer.transform(test_df)
print("---"*10)

Train: fit_transform
Re-index mapping dumped into ...
user: ../datasets/userid_mapping.csv
item: ../datasets/itemid_mapping.csv
Fitted: user/item mapping
Fitted: vocab2idx for actorID
Fitted: vocab2idx for country
Fitted: vocab2idx for directorID
Fitted: vocab2idx for genre
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Valid: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Test: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------


In [6]:
# # NOTE: can check the encoding vocab idx content from the feature engineer
# oov_idx = feature_engineer.vocab2idx["movieID"]["[OOV]"]
# len(test_df[test_df["movieID"] == oov_idx])

### Prepare Additional Data for Train/Inference

#### Build User-Item-Attributes graph for training

In [7]:
# NOTE: At training, we use interaction graph from train_df for train and validation
vocab = feature_engineer.vocab2idx
train_hetero_graph = experiment_data_preprocessor.create_knowledge_graph(encoded_train_df, vocab)


Creating interaction graph...
Drop negative samples
  Num of all interactions: 358027
  Num of positive interactions: 158984 

Building user-item edges...
Building item-attribute edges...
Knowledge Graph: HeteroData(
  user={ num_nodes=2065 },
  movie={ num_nodes=8707 },
  actor={ num_nodes=2287 },
  country={ num_nodes=42 },
  director={ num_nodes=542 },
  genre={ num_nodes=22 },
  (user, interacts_with, movie)={ edge_index=[2, 158984] },
  (movie, interacts_with, user)={ edge_index=[2, 158984] },
  (movie, has_actor, actor)={ edge_index=[2, 33520] },
  (actor, has_actor, movie)={ edge_index=[2, 33520] },
  (movie, has_country, country)={ edge_index=[2, 6704] },
  (country, has_country, movie)={ edge_index=[2, 6704] },
  (movie, has_director, director)={ edge_index=[2, 6704] },
  (director, has_director, movie)={ edge_index=[2, 6704] },
  (movie, has_genre, genre)={ edge_index=[2, 53632] },
  (genre, has_genre, movie)={ edge_index=[2, 53632] }
)
Node Type: ['user', 'movie', 'actor', '

#### Prepare train/valid triplet data

In [8]:
train_triplet_df = experiment_data_preprocessor.prepare_triplet_df(encoded_train_df, k_negative_samples=5)
valid_triplet_df = experiment_data_preprocessor.prepare_triplet_df(encoded_valid_df, k_negative_samples=5)
train_triplet_df.head(1)

Original data count (positive samples): 158984
Num of triplets: 158984(pos samples) * 5(negative sampled items) = 794920
Original data count (positive samples): 18261
Num of triplets: 18261(pos samples) * 5(negative sampled items) = 91305


,userID,pos_item_id,neg_item_id,actorID_idx,country_idx,directorID_idx,genre_idx,neg_actorID_idx,neg_country_idx,neg_directorID_idx,neg_genre_idx
0,0,1102,307,"[2267, 1401, 582, 998, 1847]",37,360,"[2, 3, 17, 18, 0, 0, 0, 0]","[1769, 713, 931, 1356, 1]",12,452,"[9, 11, 0, 0, 0, 0, 0, 0]"


#### Prepare prediction pool for inference/testing

In [9]:
# NOTE: Prepare prediction pool to evaluate the model
# valid_pool_df = experiment_data_preprocessor.prepare_prediction_df(encoded_valid_df, K=100)
prediction_pool_df = experiment_data_preprocessor.prepare_prediction_df(encoded_test_df, K=1000)
prediction_pool_df.tail()

Prediction DataFrame:
User Pool: 2064
Item Pool: 6959, negative sampled to 1000 items for each user
Num of interactions: 2064(users) * 1000(items) = 2064000


,userID,movieID,label,actorID_idx,country_idx,directorID_idx,genre_idx
2063995,2063,1660,0,"[1, 735, 1165, 1, 1]",37,1,"[4, 5, 0, 0, 0, 0, 0, 0]"
2063996,2063,2347,0,"[1, 1, 1, 1, 1]",18,134,"[4, 5, 0, 0, 0, 0, 0, 0]"
2063997,2063,4479,0,"[867, 884, 416, 2282, 1]",37,373,"[6, 17, 0, 0, 0, 0, 0, 0]"
2063998,2063,5256,0,"[865, 1756, 1810, 1, 1]",37,358,"[9, 0, 0, 0, 0, 0, 0, 0]"
2063999,2063,78,0,"[1, 1356, 1, 1, 1]",37,1,"[9, 16, 0, 0, 0, 0, 0, 0]"


### Prepare DataLoader

In [10]:
# NOTE: ensure reproducibility of DataLoader
import torch
from common.utils import seed_worker
g = torch.Generator()
g.manual_seed(RANDOM_SEED)

# TODO: determine which Dataset to use
from torch.utils.data import DataLoader
from common.datasets import TripletDataset, UserItemPairDataset

BATCH_SIZE = 1024

train_dataset = TripletDataset(train_triplet_df)
valid_dataset = TripletDataset(valid_triplet_df)
test_dataset = UserItemPairDataset(prediction_pool_df)
print("train data count:", len(train_dataset))
print("valid data count:", len(valid_dataset))
print("test data count:", len(test_dataset))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, worker_init_fn=seed_worker, generator=g, num_workers=4)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)


train data count: 794920
valid data count: 91305
test data count: 2064000


### Configure Model (LightningModule)

In [11]:
from lightning_models.extensions.kgat_v2 import KGATRecV2

EMB_DIM = 64
LR = 1e-3
EPOCHS = 50
NUM_LAYERS = 3
REG_WEIGHT = 1e-5
NUM_NEIGHBORS = 10
USE_MINI_BATCH = False

model = KGATRecV2(
    hetero_data=train_hetero_graph,
    embedding_dim=EMB_DIM,
    num_layers=NUM_LAYERS,
    num_neighbors=NUM_NEIGHBORS,
    lr=LR,
    reg_weight=REG_WEIGHT,
    use_mini_batch=USE_MINI_BATCH,
)


Seed set to 42


### Configure Trainer and Experiment

In [12]:
from common._mlflow import get_mlflow_logger, get_callbacks

EXPERIMENT_NAME = "kgat-exp"
VERSION = "v2"
RUN_NAME = "k=1000" # "run-4-mini-undirected"
PATIENCE = 5
mlflow_logger = get_mlflow_logger(experiment_name=EXPERIMENT_NAME, run_name=RUN_NAME, tags={"version": VERSION})
trainer_callbacks = get_callbacks(
    exp_name=EXPERIMENT_NAME,
    version_name=VERSION,
    run_name=RUN_NAME,
    patience=PATIENCE,
    monitor_metric="val_bpr_loss",
    monitor_mode="min",
    hyper_param_str=f"emb_dim={EMB_DIM}-num_layers={NUM_LAYERS}-lr={LR}-reg_weight={REG_WEIGHT}",
)

In [13]:
from pytorch_lightning import Trainer

trainer = Trainer(
    max_epochs=EPOCHS,
    logger=mlflow_logger,
    log_every_n_steps=50,
    callbacks=trainer_callbacks,
    accelerator='auto',  # or 'auto', 'gpu'
    # devices=[0], # if gpu is available
)


Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


### Train Model

In [14]:
# Start training
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=valid_loader)


/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /home/adam/R11_Bai/DPRecSys/experiments/test_checkpoints/kgat-exp exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name       | Type    | Params | Mode 
-----------------------------------------------
0 | kgat_model | KGAT    | 885 K  | train
1 | bpr_loss   | BPRLoss | 0      | train
2 | reg_loss   | EmbLoss | 0      | train
-----------------------------------------------
885 K     Trainable params
0         Non-trainable params
885 K     Total params
3.542     Total estimated model params size (MB)
89        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 4, global step 3885: 'val_bpr_loss' was not in top 1


Validation: |          | 0/? [00:00<?, ?it/s]

Monitored metric val_bpr_loss did not improve in the last 5 records. Best score: 3.347. Signaling Trainer to stop.
Epoch 5, global step 4662: 'val_bpr_loss' was not in top 1


🏃 View run k=1000 at: http://140.112.106.216:3683/#/experiments/11/runs/9524e984269a46b1a5c23d275632e3a4
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/11


### Inference

In [15]:
# NOTE: the inference model MUST be the same as the training model
# best_model_experiment_name = "kgat-exp"
# best_model_checkpoint_path = "run-3-full-undirected-emb_dim=128-num_layers=3-lr=0.001-reg_weight=1e-05-best-checkpoint-epoch=00-val_ndcg10=0.48.ckpt"
# best_model_path = f"test_checkpoints/{best_model_experiment_name}/{best_model_checkpoint_path}"
best_model_path = trainer.checkpoint_callback.best_model_path

model = KGATRecV2.load_from_checkpoint(checkpoint_path=best_model_path)
# start inference
trainer.test(model=model, dataloaders=test_loader)


Seed set to 42
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        test_ndcg10        │    0.23489969968795776    │
│        test_ndcg20        │    0.27620890736579895    │
│        test_ndcg5         │    0.17626643180847168    │
│     test_precision10      │    0.0702519416809082     │
│     test_precision20      │    0.06526162475347519    │
│      test_precision5      │    0.06986434012651443    │
│       test_recall10       │    0.06104832515120506    │
│       test_recall20       │    0.10718049108982086    │
│       test_recall5        │   0.032689351588487625    │
└───────────────────────────┴───────────────────────────┘

🏃 View run k=1000 at: http://140.112.106.216:3683/#/experiments/11/runs/9524e984269a46b1a5c23d275632e3a4
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/11


[{'test_ndcg5': 0.17626643180847168,
  'test_ndcg10': 0.23489969968795776,
  'test_ndcg20': 0.27620890736579895,
  'test_precision5': 0.06986434012651443,
  'test_precision10': 0.0702519416809082,
  'test_precision20': 0.06526162475347519,
  'test_recall5': 0.032689351588487625,
  'test_recall10': 0.06104832515120506,
  'test_recall20': 0.10718049108982086}]

In [16]:
model.test_results["eval_score_df"].describe()

,user,ndcg@5,recall@5,precision@5,ndcg@10,recall@10,precision@10,ndcg@20,recall@20,precision@20
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.176266,0.032689,0.069864,0.234900,0.061048,0.070252,0.276209,0.107180,0.065262
std,595.969798,0.298368,0.080326,0.119552,0.285045,0.105951,0.092453,0.254101,0.136835,0.072619
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,515.750000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1031.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.289065,0.070299,0.050000
75%,1547.250000,0.386853,0.033333,0.200000,0.430677,0.090909,0.100000,0.430677,0.162634,0.100000
max,2063.000000,1.000000,1.000000,0.600000,1.000000,1.000000,0.600000,1.000000,1.000000,0.500000


In [19]:
user_dps_df = evaluator.eval_user_diversity_preference_scale(encoded_train_df, feature_engineer.vocab2idx, normalized=True)

# Evaluate user diversity preference matching score (DPMS) at k
eval_df = evaluator.prepare_evaluation_data(model.test_results, feature_engineer.idx2vocab)

# Get user DPMS
user_dpms_df = evaluator.evaluate_dpms_at_k(
    eval_df=eval_df,
    feature_engineer=feature_engineer,
    ground_truth_dps_df=user_dps_df,
    k=10,
    actor_k=5,
    rare_threshold=5,
)

user_dpms_df.describe()

Calculating user diversity preference scale:   0%|          | 0/2064 [00:00<?, ?it/s]

Calculating user diversity preference scale: 100%|██████████| 2064/2064 [00:06<00:00, 329.78it/s]


candidate item pool size: 10
exploded 2064
extracting item features...
merging features...
interaction data count before merging: 20640
interaction data count after merging: 20640
done!
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
encoded 2064


Calculating user diversity preference scale: 100%|██████████| 2064/2064 [00:02<00:00, 715.10it/s]


combined_df 2064
Calculating DPMS for each feature...


,userID,actorID_dpms,country_dpms,directorID_dpms,genre_dpms,avg_dpms
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.129332,0.947738,0.125162,0.836496,0.509682
std,595.969798,0.067320,0.062762,0.095683,0.089502,0.053451
min,0.000000,0.000000,0.361850,0.000000,0.250516,0.261659
25%,515.750000,0.077928,0.937238,0.051201,0.794373,0.475637
50%,1031.500000,0.127084,0.970058,0.113702,0.855788,0.512803
75%,1547.250000,0.177817,0.984412,0.186993,0.900186,0.547636
max,2063.000000,0.345133,0.999864,0.558192,0.977494,0.661816


In [18]:
prefix = "kgat_v2_k5_1000"
embedding_path = "embeddings/"

torch.save(model.user_emb.cpu(), f"{embedding_path}{prefix}_user_emb.pt")
torch.save(model.item_emb.cpu(), f"{embedding_path}{prefix}_item_emb.pt")

import joblib
dir = "artifacts"
if not os.path.exists(dir):
    os.makedirs(dir)

joblib.dump(eval_df, os.path.join(dir, f"{prefix}_eval_df.pkl"))
joblib.dump(user_dps_df, os.path.join(dir, f"{prefix}_user_dps_df.pkl"))
joblib.dump(feature_engineer, os.path.join(dir, f"{prefix}_feature_engineer.pkl"))

['artifacts/kgat_v2_k5_1000_feature_engineer.pkl']